# Phase 2: Behavioral Feature Engineering & Temporal Splitting

### Transition from Business Discovery
In Phase 1, exploratory analysis revealed that customer spending behavior varies significantly, revenue is moderately spread across the user base, and simple transaction frequency alone cannot explain customer value. 

This phase transforms raw transaction data into **behavioral machine learning features** while strictly preventing data leakage.

In [9]:
import pandas as pd
import numpy as np
import warnings

# Suppress minor future warnings to keep the notebook clean
warnings.filterwarnings('ignore', category=FutureWarning)

print("--- Phase 2: Loading & Cleaning ---")
df = pd.read_csv('../data/raw/fraudTrain.csv')

# Drop fraud to focus strictly on organic behavior
df = df[df['is_fraud'] == 0]

# Convert dates
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

min_date = df['trans_date_trans_time'].min()
max_date = df['trans_date_trans_time'].max()
print(f"Dataset shape after cleaning: {df.shape[0]:,} rows")
print(f"Dataset Timeline: {min_date.date()} to {max_date.date()}")
print(df.columns.tolist())

--- Phase 2: Loading & Cleaning ---
Dataset shape after cleaning: 1,289,169 rows
Dataset Timeline: 2019-01-01 to 2020-06-21
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


### Behavioral Data Quality Validation
This project intentionally prioritizes behavioral transaction features over demographic or geographic attributes (like `lat`, `long`, `city_pop`, and `job`) to focus primarily on spending dynamics. We are predicting behavioral changes based on past actions, not demographic profiles.

In [10]:
drop_cols = ['Unnamed: 0', 'merchant', 'category', 'first', 'last', 'gender', 'street', 
             'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 
             'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

df.drop(columns=drop_cols, inplace=True)

print(f"Duplicate rows found: {df.duplicated().sum()}")
print("\nMissing values check:")

# Elite Fix: Only print columns that actually have missing values
nulls = df.isnull().sum()
missing = nulls[nulls > 0]
if missing.empty:
    print("Perfect! No missing values found.")
else:
    print(missing)
df.head()

Duplicate rows found: 0

Missing values check:
Perfect! No missing values found.


,trans_date_trans_time,cc_num,amt
0,2019-01-01 00:00:18,2703186189652095,4.97
1,2019-01-01 00:00:44,630423337322,107.23
2,2019-01-01 00:00:51,38859492057661,220.11
3,2019-01-01 00:01:16,3534093764340240,45.00
4,2019-01-01 00:03:06,375534208663984,41.96


### Preventing Data Leakage with Strict Time-Based Splitting
To simulate real-world forecasting:
* Customer behavior from **2019** is used as model input (The Observation Window).
* Customer spending from **2020** becomes the prediction target (The Future Window).

This ensures the model learns from historical behavior only and predicts truly unseen future outcomes. *(Note: This project focuses on growth prediction for retained customers rather than churn prediction).*

In [12]:
cutoff_date = pd.to_datetime('2019-12-31')

print(f"Splitting data strictly at cutoff date: {cutoff_date.date()}")

# 1. Split into Past (Features) and Future (Target)
feature_window = df[df['trans_date_trans_time'] <= cutoff_date]
target_window = df[df['trans_date_trans_time'] > cutoff_date]

# 2. Prevent Data Leakage: Only keep customers who exist in BOTH windows
users_in_features = set(feature_window['cc_num'].unique())
users_in_target = set(target_window['cc_num'].unique())
valid_users = users_in_features.intersection(users_in_target)

feature_window = feature_window[feature_window['cc_num'].isin(valid_users)]
target_window = target_window[target_window['cc_num'].isin(valid_users)]

print("\n--- Time Window Split Successful ---")
print(f"Observation Window (Features): {len(feature_window):,} rows")
print(f"Target Window (Predictions): {len(target_window):,} rows")
print(f"Total valid, retained users: {len(valid_users):,}")

Splitting data strictly at cutoff date: 2019-12-31

--- Time Window Split Successful ---
Observation Window (Features): 916,757 rows
Target Window (Predictions): 372,412 rows
Total valid, retained users: 908


In [13]:
display(feature_window.head())
display(target_window.head())

,trans_date_trans_time,cc_num,amt
0,2019-01-01 00:00:18,2703186189652095,4.97
1,2019-01-01 00:00:44,630423337322,107.23
2,2019-01-01 00:00:51,38859492057661,220.11
3,2019-01-01 00:01:16,3534093764340240,45.00
4,2019-01-01 00:03:06,375534208663984,41.96


,trans_date_trans_time,cc_num,amt
921954,2019-12-31 00:00:22,3546674063249004,42.55
921955,2019-12-31 00:01:13,3598014571045296,125.68
921956,2019-12-31 00:01:36,630469040731,155.62
921957,2019-12-31 00:03:51,4890424426862856940,45.22
921958,2019-12-31 00:03:56,4681699462969,40.02


### Elite Feature Engineering
We transform raw rows into mathematical behavioral profiles. 
* **Target Transformation:** The raw growth ratio is engineered here for interpretability. During predictive modeling (Phase 4), a logarithmic transformation is applied to stabilize extreme financial behavior and improve model robustness.

### Window Length Consideration & Target Definition
The historical observation window (2019) covers a full 12-month span, whereas the future prediction window (2020) covers approximately 6 months (Jan-June). Because we do not annualize the 2020 data, the engineered `spend_growth_rate` ratio should be strictly interpreted as **Relative Future Spending Intensity** during the pandemic volatility period, rather than a pure year-over-year 12-month growth metric.

In [19]:
print("Calculating Advanced Behavioral Features...")

# 1. Target Variable (2020 Spend)
target_df = target_window.groupby('cc_num')['amt'].sum().reset_index()
target_df.rename(columns={'amt': 'future_spend'}, inplace=True)
display(target_df.head())

# 2. Engineered Behavioral Features (2019 Base)
rfm = feature_window.groupby('cc_num').agg(
    frequency=('amt', 'count'),
    monetary_sum=('amt', 'sum'),
    monetary_mean=('amt', 'mean'),
    monetary_max=('amt', 'max'),
    monetary_std=('amt', 'std'),
    first_swipe=('trans_date_trans_time', 'min'),
    last_swipe=('trans_date_trans_time', 'max')
).reset_index()

# Handle standard deviation NaNs for customers with only 1 transaction
rfm['monetary_std'] = rfm['monetary_std'].fillna(0)

# 3. Time-based behavioral metrics
rfm['active_duration_days'] = (rfm['last_swipe'] - rfm['first_swipe']).dt.days
rfm['avg_days_between_swipes'] = rfm['active_duration_days'] / np.maximum(rfm['frequency'] - 1, 1)

# ELITE ADDITION: Recency (Days since last swipe relative to the end of 2019)
# cutoff_date = feature_window['trans_date_trans_time'].max()
# rfm['recency_days'] = (cutoff_date - rfm['last_swipe']).dt.days

# Drop raw dates
rfm.drop(columns=['first_swipe', 'last_swipe'], inplace=True)

# 4. Merge Features with Target
ml_df = pd.merge(rfm, target_df, on='cc_num', how='inner')

# 5. Calculate Raw Growth Rate (Interpretability)
ml_df['spend_growth_rate'] = ml_df['future_spend'] / ml_df['monetary_sum']

print("\nSUCCESS! Elite Machine Learning Dataset is ready.")
print(f"Total Dataset Variables: {len(ml_df.columns)}")
display(ml_df.head())

print("\n--- Feature Statistical Summary ---")
display(ml_df.describe().round(2))

Calculating Advanced Behavioral Features...


,cc_num,future_spend
0,60416207185,22598.03
1,60422928733,28573.45
2,60423098130,34268.53
3,60427851591,14366.46
4,60487002085,9461.55



SUCCESS! Elite Machine Learning Dataset is ready.
Total Dataset Variables: 10


,cc_num,frequency,monetary_sum,monetary_mean,monetary_max,monetary_std,active_duration_days,avg_days_between_swipes,future_spend,spend_growth_rate
0,60416207185,1090,60545.71,55.546523,3075.09,123.736332,363,0.333333,22598.03,0.373239
1,60422928733,1098,69566.57,63.357532,1290.37,81.188377,361,0.329079,28573.45,0.410735
2,60423098130,367,19807.02,53.970082,455.32,64.296167,363,0.991803,34268.53,1.730120
3,60427851591,371,35117.36,94.655957,569.40,86.925398,363,0.981081,14366.46,0.409099
4,60487002085,316,15698.56,49.678987,475.99,56.672238,358,1.136508,9461.55,0.602702



--- Feature Statistical Summary ---


,cc_num,frequency,monetary_sum,monetary_mean,monetary_max,monetary_std,active_duration_days,avg_days_between_swipes,future_spend,spend_growth_rate
count,9.080000e+02,908.00,908.00,908.00,908.00,908.00,908.00,908.00,908.00,908.00
mean,4.090328e+17,1009.64,68302.80,67.57,2436.00,127.38,362.56,0.50,27770.26,0.41
std,1.293983e+18,532.55,40204.68,14.39,2885.77,90.28,1.10,0.31,16607.85,0.08
min,6.041621e+10,286.00,15698.56,44.99,297.44,50.33,353.00,0.16,4917.67,0.15
25%,1.800391e+14,684.25,37309.38,57.28,934.99,84.92,363.00,0.25,14869.69,0.37
50%,3.521616e+15,1062.00,64186.82,62.82,1533.64,105.43,363.00,0.34,25632.97,0.40
75%,4.651725e+15,1440.00,90097.96,75.00,2715.93,135.23,363.00,0.53,36406.02,0.44
max,4.992346e+18,2238.00,213772.62,125.93,28948.90,1404.59,363.00,1.27,89129.54,1.73


,cc_num,future_spend
0,60416207185,22598.03
1,60422928733,28573.45
2,60423098130,34268.53
3,60427851591,14366.46
4,60487002085,9461.55


### Feature Validation & Behavioral Diagnostics
No single feature strongly explains future growth independently, proving that simple rules-based logic cannot predict CLV. This supports the need for non-linear Machine Learning models.

In [17]:
print("--- Correlation with Spend Growth Rate ---")

# Drop Customer ID before calculating correlations
correlations = ml_df.drop(columns=['cc_num']).corr()
print(correlations['spend_growth_rate'].sort_values(ascending=False))

# Save the pristine dataset for K-Means and XGBoost
ml_df.to_csv('../data/processed/ml_features.csv', index=False)
print("\n✅ Success! Engineered features saved to ../data/processed/ml_features.csv")

--- Correlation with Spend Growth Rate ---
spend_growth_rate          1.000000
future_spend               0.118894
avg_days_between_swipes    0.059698
active_duration_days      -0.024790
frequency                 -0.050843
monetary_sum              -0.080302
monetary_mean             -0.165476
monetary_max              -0.271172
monetary_std              -0.317858
Name: spend_growth_rate, dtype: float64

✅ Success! Engineered features saved to ../data/processed/ml_features.csv
